# 🔐 Sistema de Seguridad Facial con Clasificación de Zonas

**Trabajo Final — Computer Vision**  
Master en Deep Learning · MIOTI · 2026  
- Alvaro Marro
- Mynor Hernandez
- Juanmanuel Piedrabuena

---

## 1. Motivación

En muchos hogares ya existen cámaras de seguridad instaladas, pero el software que las gestiona suele ser de pago o estar limitado a funciones básicas como grabación continua o detección de movimiento. La pregunta que guió este proyecto fue directa: **con las herramientas de deep learning que hemos visto en el máster, ¿es posible construir un sistema equivalente, de código abierto y ejecutable en local?**

La respuesta fue sí, y el objetivo se convirtió en ir más allá de la detección de movimiento simple: construir un sistema que reconozca quiénes son las personas autorizadas, detecte intrusos y además informe en qué zona del hogar fueron detectados — todo en tiempo real y con notificación al móvil.

Un aspecto que resultó sorprendente durante el desarrollo fue que **la whitelist de personas autorizadas se implementa exactamente con el mismo mecanismo que vimos en NLP**: embeddings y similitud coseno. ArcFace genera un vector de 512 dimensiones por cara, igual que un modelo de lenguaje genera embeddings de palabras. La analogía no es casual — es la misma matemática aplicada a otro dominio.

---

## 2. Arquitectura del sistema

El sistema combina tres componentes principales que se ejecutan en secuencia sobre cada frame de la cámara:

```
Frame de cámara
      ↓
YOLOv12n-face → detecta caras
      ↓
InsightFace (ArcFace) → ¿está en la whitelist? (similitud coseno, umbral 0.5)
      ↓
ConvNeXt-Small (fastai) → ¿en qué zona está?
      ↓
"🚨 Intruso detectado en COCINA"
      ↓
Alerta push (ntfy) + captura guardada en alerts_log/
```

| Componente | Tecnología |
|---|---|
| Detección facial | YOLOv12n-face — modelo YOLO específico para caras |
| Reconocimiento | InsightFace (ArcFace) — embeddings 512d + similitud coseno |
| Clasificación de zona | ConvNeXt-Small fine-tuned (fastai) sobre MIT Indoor Scenes |
| Alertas | ntfy.sh — notificación push al móvil, gratuito |
| Panel de control | Gradio — gestión de whitelist y log de alertas |

Algo que no esperaba: YOLO tiene versiones específicas preentrenadas para tareas concretas — detección de caras, matrículas, gestos. Usamos `yolov12n-face.pt`, un modelo nano optimizado para caras que corre a tiempo real en CPU sin problemas. Esta especialización mejora notablemente la precisión frente al modelo genérico.

---

## 3. Clasificador de zonas

### 3.1 Dataset

Para el clasificador usé el **MIT Indoor Scenes dataset (CVPR 2009)**, un dataset académico de referencia con 67 categorías de escenas interiores. Filtré las 8 clases relevantes para un entorno de seguridad doméstica:

```
bathroom · bedroom · corridor · dining_room
garage · kitchen · livingroom · stairscase
```

La decisión de filtrar en vez de usar las 67 clases fue deliberada: un clasificador con categorías como "aeropuerto" o "casino" no aporta nada en este contexto. El dataset presenta desbalance notable (kitchen: 734 imágenes vs garage: 103), gestionado con **split estratificado**.

### 3.2 Entrenamiento

**Arquitectura:** ConvNeXt-Small preentrenado en ImageNet-22k (transfer learning).  
**Learning rate:** 1e-2, elegido con `lr_find()`.  
**Regularización aplicada para combatir overfitting:**

- **MixUp (alpha=0.3):** mezcla pares de imágenes durante el entrenamiento. El modelo aprende distribuciones más suaves y generaliza mejor.
- **Label Smoothing:** en vez de optimizar hacia 1.0 exacto, suaviza las etiquetas. Reduce la sobreconfianza en las predicciones.
- **EarlyStoppingCallback (patience=3):** parada automática cuando el `error_rate` deja de mejorar.

### 3.3 Resultados

| Métrica | Valor |
|---|---|
| Accuracy final | **95.1%** |
| Error rate | **0.048** |
| Épocas entrenadas | 12 (early stopping, mejor modelo en época 10) |
| Mejora vs sin regularización | +3.6% accuracy (91.5% → 95.1%) |
| Arquitectura | ConvNeXt-Small (ImageNet-22k) |

El impacto de MixUp + Label Smoothing fue claro y medible: sin ellas el modelo llegaba al 91.5% con overfitting visible desde la época 4. Con regularización el `valid_loss` continuó descendiendo de forma consistente hasta la parada automática.

### 3.4 Análisis de errores

La matriz de confusión revela que los errores no son aleatorios — se concentran en pares de clases visualmente similares:

- **bedroom / livingroom:** 7 confusiones. Ambas tienen muebles tapizados e iluminación similar.
- **dining_room / kitchen:** errores razonables dado que comparten superficies y sillas.
- **garage:** clasificación perfecta. Visualmente muy distinto al resto.

El análisis de `top_losses` confirma esta lectura: los peores errores corresponden a imágenes genuinamente ambíguas — una sala con futón que parece dormitorio, una cocina con chimenea que parece salón. En algún caso el modelo parece tener razón y el label del dataset original es el que está mal etiquetado.

**Conclusión:** el modelo falla donde un humano también dudaría. El límite de precisión no es un problema de arquitectura ni de entrenamiento, sino de ambigüedad inherente al dataset.

---

## 4. Integración del pipeline

La integración de ambos modelos en el sistema de seguridad fue directa. El loop principal captura frames de la cámara, pasa cada frame por InsightFace para reconocimiento y por ConvNeXt para clasificación de zona. Ambas inferencias son independientes y se ejecutan en cada frame.

La notificación push incluye la zona detectada: `"Intruso en COCINA"` en vez de solo `"Intruso detectado"`. Esto da contexto útil — no es lo mismo un intruso en el garaje que en el dormitorio.

Las capturas guardadas en `alerts_log/` incluyen el bounding box, el score de reconocimiento y la zona superpuesta en el frame. El sistema incluye un **cooldown de 10 segundos** por intruso para evitar spam de alertas.

---

## 5. Conclusiones

El proyecto demuestra que es posible construir un sistema de seguridad con reconocimiento facial y clasificación de escena de calidad comparable a soluciones comerciales, usando únicamente herramientas open source y hardware consumer (Mac M1, cámara integrada).

Los aprendizajes más relevantes del proceso:

- **La transferencia de conceptos entre dominios es real:** embeddings + similitud coseno funciona igual para texto que para caras. Verlo aplicado en CV refuerza la comprensión del mecanismo.
- **YOLO no es un único modelo:** el ecosistema incluye versiones especializadas para tareas específicas (caras, matrículas, gestos) que superan al modelo genérico en su dominio.
- **MixUp y Label Smoothing tienen impacto medible:** +3.6% de accuracy en un escenario con dataset limitado y desbalanceado. No son opcionales en estos contextos.
- **Los errores del modelo son informativos:** cuando los fallos se concentran en clases ambiguas para humanos, el cuello de botella es el dataset, no la arquitectura.

---

## 6. Referencias y enlaces

| Recurso | Enlace |
|---|---|
| Código fuente | https://github.com/Mynozy/CV_ProyectoFinal |
| Modelo en HuggingFace | https://huggingface.co/mynorhm/security-room-classifier |
| Dataset | MIT Indoor Scenes CVPR 2009 |
| InsightFace | https://github.com/deepinsight/insightface |
| YOLO-face | https://github.com/akanametov/yolo-face |
